# Llama 3.2 Vision image evaluation notebook

## Install and import required dependencies

First, let's install the necessary packages for running Llama 3.2 Vision locally.

In [3]:
# Install required packages
# !pip install transformers torch torchvision pillow requests

# # Optional: Install additional packages for enhanced functionality
# !pip install matplotlib ipywidgets tqdm

In [24]:
import os
from tqdm import tqdm
import torch
from PIL import Image
from transformers import MllamaForConditionalGeneration, AutoProcessor
from datetime import datetime
import pandas as pd

## Set up model

In [3]:
# Model path configuration
MODEL_PATH = "/home/prateek/models/Llama-3.2-11B-Vision-Instruct"

# Load model and processor
model = MllamaForConditionalGeneration.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

processor = AutoProcessor.from_pretrained(MODEL_PATH)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

## Define functions

In [27]:
# load an image from a file path
def load_image(image_path: str) -> Image.Image:
    img = Image.open(image_path)
    return img

# create a function to evaluate an image with a text prompt
def evaluate_image_with_prompt(image: Image.Image, prompt: str, max_new_tokens: int) -> str:
    """
    Send an image and text prompt to Llama for analysis.
    Args:
        image: PIL Image object
        prompt: Text prompt for zero-shot analysis
    Returns:
        Analysis result as string
    """
    # Prepare messages
    messages = [
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": prompt}
        ]}
    ]
    
    # Process inputs
    input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(
        image,
        input_text,
        add_special_tokens=False,
        return_tensors="pt"
    ).to(model.device)
    
    # Generate response
    output = model.generate(**inputs, max_new_tokens=max_new_tokens)
    
    # Decode result
    result = processor.decode(output[0], 
                skip_special_tokens=False, 
                clean_up_tokenization_spaces=False
            )
    
    # Extract only the response part
    if input_text in result:
        result = result.replace(input_text, "").strip()
    
    return result

## Example usage - single image analysis

Let's test our setup with a single image.

In [ ]:
test_image_path = "/projects/matsci/vlm_microscopy/Microscopy/NFFA/sampled_data/Biological/1a88ebda668e2e7650eddd6f65de8876.jpg"
test_prompt = "Describe the image."

evaluate_image_with_prompt(load_image(test_image_path), test_prompt)

'The image presents a detailed microscopic view of a surface, with a white bar at the bottom providing information about the image. The purpose of the image is to display the surface\'s texture and features.\n\n* A microscopic view of a surface:\n\t+ The surface appears to be rough and textured.\n\t+ It has a complex structure with many small features.\n\t+ The surface is likely made of a material that is not visible to the naked eye.\n* A white bar at the bottom of the image:\n\t+ The bar contains various labels and measurements.\n\t+ The labels include "EHT", "WD", "Signal A", "Stage at T", "Stage at Z", "Mag", "Brightness", and "Contrast".\n\t+ The measurements include 1 μm, 50.00 KX, 4 mm, 0.0°, 49.000 mm, 30.00 μm, 48.2%, and 47.9%.\n* A logo in the bottom-right corner of the image:\n\t+ The logo is small and located in the corner of the image.\n\t+ It appears to be a stylized letter "T" or "TASC".\n\nOverall, the image provides a detailed view of a surface\'s texture and features

## Batch processing of images

Prepare images and prompt for batch processing.

In [41]:
# provide a list of directories containing images
image_dirs = [
    "/projects/matsci/vlm_microscopy/Microscopy/NFFA/sampled_counting_cropped/Particles",
]

# provide a list of allowed image extensions
allowed_extensions = [".jpg", ".jpeg", ".png", ".bmp", ".tiff"]

# get a list of all image file paths recursively in the directories
image_paths = []
for directory in image_dirs:
    for root, _, files in os.walk(directory):
        for file in files:
            if any(file.lower().endswith(ext) for ext in allowed_extensions):
                image_paths.append(os.path.join(root, file))

print(f"Found {len(image_paths)} images for analysis.")

# set your prompt
PROMPT = "Can you visually estimate the number of particles?"

# set the maximum number of tokens for generation
MAX_TOKENS = 100

# other prompts for reference
# PROMPT = "Describe this image."

Found 25 images for analysis.


Run the batch.

In [42]:
# Prepare results list
results = []

for img_path in tqdm(image_paths):
    img = load_image(img_path)
    result = evaluate_image_with_prompt(img, PROMPT, max_new_tokens=MAX_TOKENS)
    # Store result
    results.append({
        'image_path': img_path,
        'image_name': os.path.basename(img_path),
        'prompt': PROMPT,
        'llama_response': result,
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'status': 'success'
    })

df = pd.DataFrame(results)
# Save to CSV
df.to_csv('llama_vqa_t2_count_particles.csv', index=False)

100%|██████████| 25/25 [00:47<00:00,  1.91s/it]


## 🎉 Ready to Start!

You now have a complete setup for using Llama 3.2 Vision locally for image analysis. Here's what to do next:

### 🚀 Quick Start Checklist:

1. **✅ Ensure model is downloaded** (Section 3)
2. **✅ Load an image** (Section 7) - Use URL or local file
3. **✅ Try different prompts** - Use templates or create custom ones
4. **✅ Experiment with settings** - Adjust prompts and test different images

### 📝 Common Next Steps:

- **Replace the test image URL** with your own images
- **Try different prompt templates** from the `prompt_templates` dictionary
- **Experiment with batch processing** for multiple images
- **Use interactive mode** for real-time testing
- **Integrate with your own datasets** and workflows

### ? Local Model Advantages:

- **Privacy**: All processing happens locally
- **No Rate Limits**: Process as many images as your hardware allows
- **Offline**: Works without internet connection
- **Performance**: Optimized for your hardware

### ? Need Help?

- Check that the model is properly downloaded to `/home/prateek/models/`
- Use the diagnostic functions to troubleshoot issues
- Start with simple prompts and gradually increase complexity
- Monitor your system resources during processing

**Happy analyzing! 🎨📸🤖**